# PySpark practice 10: reading and writing data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-10-reading-and-writing.ipynb)

14 exercises on readers and options, corrupt records, partitioned and sized writes, tables, and Delta MERGE, time travel and schema evolution. Each exercise names the [reference entries](pyspark-reference-10-reading-and-writing.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 10](pyspark-reference-10-reading-and-writing.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell. The session loads Delta Lake; `files_in` and `write_text` are file helpers.

In [ ]:
import os
import sys
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the examples write

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import glob
import os


def files_in(path, pattern="*.parquet"):
    """Data files under path (recursively), for showing what a write produced."""
    return sorted(os.path.relpath(p, path) for p in glob.glob(os.path.join(path, "**", pattern), recursive=True))


def write_text(path, text):
    """Write a small raw file, the way a source system would drop it."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text)

## Exercises

5 easy, 8 medium, 1 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Read a semicolon CSV with an explicit schema](#exercise-1-read-a-semicolon-csv-with-an-explicit-schema) | Easy | [CSV options](pyspark-reference-10-reading-and-writing.ipynb#csv-options) |
| 2 | [Quarantine malformed rows](#exercise-2-quarantine-malformed-rows) | Medium | [Malformed records: `mode` and `_corrupt_record`](pyspark-reference-10-reading-and-writing.ipynb#malformed-records-mode-and-_corrupt_record) |
| 3 | [Read a multi-line JSON document](#exercise-3-read-a-multi-line-json-document) | Easy | [JSON, Parquet and ORC](pyspark-reference-10-reading-and-writing.ipynb#json-parquet-and-orc) |
| 4 | [Merge Parquet schemas across files](#exercise-4-merge-parquet-schemas-across-files) | Medium | [JSON, Parquet and ORC](pyspark-reference-10-reading-and-writing.ipynb#json-parquet-and-orc) |
| 5 | [Rows per source file](#exercise-5-rows-per-source-file) | Easy | [File metadata: the `_metadata` column](pyspark-reference-10-reading-and-writing.ipynb#file-metadata-the-_metadata-column) |
| 6 | [Partitioned write and pruning](#exercise-6-partitioned-write-and-pruning) | Medium | [`partitionBy` and partition pruning](pyspark-reference-10-reading-and-writing.ipynb#partitionby-and-partition-pruning) |
| 7 | [Reload one day without touching the others](#exercise-7-reload-one-day-without-touching-the-others) | Medium | [Dynamic partition overwrite](pyspark-reference-10-reading-and-writing.ipynb#dynamic-partition-overwrite) |
| 8 | [Cap rows per output file](#exercise-8-cap-rows-per-output-file) | Easy | [Controlling file count and size](pyspark-reference-10-reading-and-writing.ipynb#controlling-file-count-and-size) |
| 9 | [insertInto matches columns by position](#exercise-9-insertinto-matches-columns-by-position) | Medium | [Tables: `saveAsTable` and `insertInto`](pyspark-reference-10-reading-and-writing.ipynb#tables-saveastable-and-insertinto) |
| 10 | [Upsert customer changes with MERGE](#exercise-10-upsert-customer-changes-with-merge) | Medium | [`MERGE`: upserts with `DeltaTable.merge`](pyspark-reference-10-reading-and-writing.ipynb#merge-upserts-with-deltatablemerge) |
| 11 | [Merge a feed that contains duplicate keys](#exercise-11-merge-a-feed-that-contains-duplicate-keys) | Hard | [`MERGE`: upserts with `DeltaTable.merge`](pyspark-reference-10-reading-and-writing.ipynb#merge-upserts-with-deltatablemerge) |
| 12 | [Read an earlier version with time travel](#exercise-12-read-an-earlier-version-with-time-travel) | Easy | [Writing and reading Delta tables, history and time travel](pyspark-reference-10-reading-and-writing.ipynb#writing-and-reading-delta-tables-history-and-time-travel) |
| 13 | [Replace one day in a Delta table](#exercise-13-replace-one-day-in-a-delta-table) | Medium | [Replace part of a table: `replaceWhere`](pyspark-reference-10-reading-and-writing.ipynb#replace-part-of-a-table-replacewhere) |
| 14 | [Evolve a Delta schema on purpose](#exercise-14-evolve-a-delta-schema-on-purpose) | Medium | [Schema enforcement and evolution](pyspark-reference-10-reading-and-writing.ipynb#schema-enforcement-and-evolution) |

## Exercise 1: Read a semicolon CSV with an explicit schema

**Difficulty:** Easy · about 5 min · **Practises:** [CSV options](pyspark-reference-10-reading-and-writing.ipynb#csv-options)

*Vendor files rarely use commas, ISO dates or empty strings for nulls.*

Read `vendor.csv` (semicolon separated, `-` means null, dates `dd.MM.yyyy`) with schema `sku string, price decimal(8,2), listed date`. Return all columns ordered by `sku`.

In [ ]:
write_text(f"{WORK_DIR}/p10/vendor.csv", "sku;price;listed\nSKU-2;19.99;03.01.2026\nSKU-1;5.50;01.01.2026\nSKU-3;-;02.01.2026\n")
print(open(f"{WORK_DIR}/p10/vendor.csv").read())

**Expected output**

| sku | price | listed |
| --- | --- | --- |
| SKU-1 | 5.50 | 2026-01-01 |
| SKU-2 | 19.99 | 2026-01-03 |
| SKU-3 | null | 2026-01-02 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (spark.read.schema("sku string, price decimal(8,2), listed date")
          .options(header=True, sep=";", nullValue="-", dateFormat="dd.MM.yyyy")
          .csv(f"{WORK_DIR}/p10/vendor.csv").orderBy("sku"))
result.show()

## Exercise 2: Quarantine malformed rows

**Difficulty:** Medium · about 10 min · **Practises:** [Malformed records: `mode` and `_corrupt_record`](pyspark-reference-10-reading-and-writing.ipynb#malformed-records-mode-and-_corrupt_record)

*Bronze layers keep every row and route the bad ones to a quarantine table.*

Read `dirty.csv` in PERMISSIVE mode with a corrupt-record column. Return `kind` (`good`, `bad`) and `rows`, ordered by `kind`.

In [ ]:
write_text(f"{WORK_DIR}/p10/dirty.csv", "id,qty\n1,4\n2,x\n3,5,extra\n4,7\n")
print(open(f"{WORK_DIR}/p10/dirty.csv").read())

**Expected output**

| kind | rows |
| --- | --- |
| bad | 2 |
| good | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
raw = (spark.read.schema("id int, qty int, _corrupt_record string").option("header", True)
       .option("columnNameOfCorruptRecord", "_corrupt_record").csv(f"{WORK_DIR}/p10/dirty.csv").cache())
result = raw.groupBy(F.when(F.col("_corrupt_record").isNull(), "good").otherwise("bad").alias("kind")).agg(F.count("*").alias("rows")).orderBy("kind")
result.show()

## Exercise 3: Read a multi-line JSON document

**Difficulty:** Easy · about 5 min · **Practises:** [JSON, Parquet and ORC](pyspark-reference-10-reading-and-writing.ipynb#json-parquet-and-orc)

*APIs often return one pretty-printed JSON array per file, not JSON Lines.*

Read `export.json` (a JSON array spanning several lines). Return `id` and `type`, ordered by `id`.

In [ ]:
write_text(f"{WORK_DIR}/p10/export.json", '[\n  {"id": 2, "type": "buy"},\n  {"id": 1, "type": "view"}\n]')
print(open(f"{WORK_DIR}/p10/export.json").read())

**Expected output**

| id | type |
| --- | --- |
| 1 | view |
| 2 | buy |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.read.option("multiLine", True).json(f"{WORK_DIR}/p10/export.json").select("id", "type").orderBy("id")
result.show()

## Exercise 4: Merge Parquet schemas across files

**Difficulty:** Medium · about 10 min · **Practises:** [JSON, Parquet and ORC](pyspark-reference-10-reading-and-writing.ipynb#json-parquet-and-orc)

*A source added a column halfway through the month; old files lack it.*

Read both batches under `evolving` with schema merging. Return `id`, `code`, `score`, ordered by `id`.

In [ ]:
base = f"{WORK_DIR}/p10/evolving"
spark.createDataFrame([(1, "a")], "id int, code string").write.mode("overwrite").parquet(f"{base}/batch=1")
spark.createDataFrame([(2, "b", 9.5)], "id int, code string, score double").write.mode("overwrite").parquet(f"{base}/batch=2")

**Expected output**

| id | code | score |
| --- | --- | --- |
| 1 | a | null |
| 2 | b | 9.5 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.read.option("mergeSchema", True).parquet(base).select("id", "code", "score").orderBy("id")
result.show()

## Exercise 5: Rows per source file

**Difficulty:** Easy · about 5 min · **Practises:** [File metadata: the `_metadata` column](pyspark-reference-10-reading-and-writing.ipynb#file-metadata-the-_metadata-column)

*Lineage: which file did each row come from?*

Write `orders` as CSV with one file per `status` (repartition by status). Read it back and return `status` and `files` (distinct source file names per status), ordered by `status`.

In [ ]:
lineage = f"{WORK_DIR}/p10/lineage"

**Expected output**

| status | files |
| --- | --- |
| paid | 1 |
| placed | 1 |
| returned | 1 |
| shipped | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
orders.repartition(4, "status").write.mode("overwrite").option("header", True).csv(lineage)
back = spark.read.option("header", True).csv(lineage)
result = back.groupBy("status").agg(F.count_distinct("_metadata.file_name").alias("files")).orderBy("status")
result.show()

## Exercise 6: Partitioned write and pruning

**Difficulty:** Medium · about 10 min · **Practises:** [`partitionBy` and partition pruning](pyspark-reference-10-reading-and-writing.ipynb#partitionby-and-partition-pruning)

*Date-partitioned tables let readers skip most of the data.*

Write `orders` partitioned by UTC `day`. Return `day_dirs` (number of partition directories) and `pruned_read_has_partition_filter` (whether reading `day = '2026-01-05'` shows the day in `PartitionFilters`).

In [ ]:
part_path = f"{WORK_DIR}/p10/by_day"

**Expected output**

| day_dirs | pruned_read_has_partition_filter |
| --- | --- |
| 8 | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
orders.withColumn("day", F.to_date("order_ts")).write.mode("overwrite").partitionBy("day").parquet(part_path)
dirs = [d for d in os.listdir(part_path) if d.startswith("day=")]
plan = "\n".join(plan_lines(spark.read.parquet(part_path).where("day = '2026-01-05'"), "formatted"))
partition_line = next(l for l in plan.splitlines() if "PartitionFilters" in l)
result = spark.createDataFrame([(len(dirs), "day" in partition_line)], "day_dirs int, pruned_read_has_partition_filter boolean")
result.show()

## Exercise 7: Reload one day without touching the others

**Difficulty:** Medium · about 15 min · **Practises:** [Dynamic partition overwrite](pyspark-reference-10-reading-and-writing.ipynb#dynamic-partition-overwrite)

*Idempotent daily reloads replace only the recomputed partition.*

Write `orders` partitioned by `day`. Then overwrite only 2026-01-12 with amounts doubled using dynamic partition overwrite, keeping `amount` as `decimal(12,2)`. Return `day` and `revenue` from the final table, ordered by `day`. Why does the type matter here?

In [ ]:
reload_path = f"{WORK_DIR}/p10/reload"

**Expected output**

| day | revenue |
| --- | --- |
| 2026-01-05 | 234.50 |
| 2026-01-06 | 260.00 |
| 2026-01-07 | 204.25 |
| 2026-01-09 | 310.00 |
| 2026-01-10 | 89.90 |
| 2026-01-12 | 2508.00 |
| 2026-01-13 | 210.00 |
| 2026-01-14 | 96.40 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
daily = orders.withColumn("day", F.to_date("order_ts"))
daily.write.mode("overwrite").partitionBy("day").parquet(reload_path)
fix = daily.where("day = '2026-01-12'").withColumn("amount", (F.col("amount") * 2).cast("decimal(12,2)"))
fix.write.mode("overwrite").option("partitionOverwriteMode", "dynamic").partitionBy("day").parquet(reload_path)
result = spark.read.parquet(reload_path).groupBy("day").agg(F.sum("amount").alias("revenue")).orderBy("day")
result.show()

## Exercise 8: Cap rows per output file

**Difficulty:** Easy · about 5 min · **Practises:** [Controlling file count and size](pyspark-reference-10-reading-and-writing.ipynb#controlling-file-count-and-size)

*Downstream systems limit file sizes; maxRecordsPerFile splits large outputs.*

Write `order_items` as one task with `maxRecordsPerFile` = 4. Return `files` (Parquet files written) and `rows` (total rows read back).

In [ ]:
cap_path = f"{WORK_DIR}/p10/capped"

**Expected output**

| files | rows |
| --- | --- |
| 5 | 17 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
order_items.coalesce(1).write.mode("overwrite").option("maxRecordsPerFile", 4).parquet(cap_path)
result = spark.createDataFrame([(len(files_in(cap_path)), spark.read.parquet(cap_path).count())], "files int, rows int")
result.show()

## Exercise 9: insertInto matches columns by position

**Difficulty:** Medium · about 10 min · **Practises:** [Tables: `saveAsTable` and `insertInto`](pyspark-reference-10-reading-and-writing.ipynb#tables-saveastable-and-insertinto)

*A silent column swap is one of the nastiest production bugs.*

Create table `p10_prices (sku string, region string)`. Insert `incoming` (columns in the order `region, sku`) correctly by reordering to the table's columns. Return the table ordered by `sku`.

In [ ]:
spark.sql("DROP TABLE IF EXISTS p10_prices")
spark.createDataFrame([], "sku string, region string").write.format("parquet").saveAsTable("p10_prices")
incoming = spark.createDataFrame([("EU", "SKU-1"), ("US", "SKU-2")], "region string, sku string")
incoming.show()

**Expected output**

| sku | region |
| --- | --- |
| SKU-1 | EU |
| SKU-2 | US |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
incoming.select(spark.table("p10_prices").columns).write.insertInto("p10_prices")
result = spark.table("p10_prices").orderBy("sku")
result.show()

## Exercise 10: Upsert customer changes with MERGE

**Difficulty:** Medium · about 15 min · **Practises:** [`MERGE`: upserts with `DeltaTable.merge`](pyspark-reference-10-reading-and-writing.ipynb#merge-upserts-with-deltatablemerge)

*CDC upserts into Delta are the core of most lakehouse pipelines.*

Merge `changes` into the Delta table at `cust_path` on `customer_id`: update matched rows only when the change is newer (`updated_at`), insert new customers. Return `customer_id`, `tier`, `updated_at`, ordered.

In [ ]:
from delta.tables import DeltaTable
cust_path = f"{WORK_DIR}/p10/delta_customers"
spark.createDataFrame([(1, "gold", datetime(2026, 1, 1)), (2, "silver", datetime(2026, 1, 5))], "customer_id int, tier string, updated_at timestamp").write.format("delta").mode("overwrite").save(cust_path)
changes = spark.createDataFrame([(1, "platinum", datetime(2026, 1, 3)), (2, "bronze", datetime(2026, 1, 2)), (3, "silver", datetime(2026, 1, 4))], "customer_id int, tier string, updated_at timestamp")
changes.show()

**Expected output**

| customer_id | tier | updated_at |
| --- | --- | --- |
| 1 | platinum | 2026-01-03 00:00:00 |
| 2 | silver | 2026-01-05 00:00:00 |
| 3 | silver | 2026-01-04 00:00:00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
(DeltaTable.forPath(spark, cust_path).alias("t")
 .merge(changes.alias("s"), "t.customer_id = s.customer_id")
 .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
 .whenNotMatchedInsertAll()
 .execute())
result = spark.read.format("delta").load(cust_path).orderBy("customer_id")
result.show()

## Exercise 11: Merge a feed that contains duplicate keys

**Difficulty:** Hard · about 15 min · **Practises:** [`MERGE`: upserts with `DeltaTable.merge`](pyspark-reference-10-reading-and-writing.ipynb#merge-upserts-with-deltatablemerge)

*MERGE fails when several source rows match one target row; production feeds often contain retries.*

`feed` has two versions of customer 1. Keep only the latest version per key (by `updated_at`), merge it into the Delta table at `dup_path`, and return the table ordered by `customer_id`.

In [ ]:
from delta.tables import DeltaTable
dup_path = f"{WORK_DIR}/p10/delta_dups"
spark.createDataFrame([(1, "gold", datetime(2026, 1, 1))], "customer_id int, tier string, updated_at timestamp").write.format("delta").mode("overwrite").save(dup_path)
feed = spark.createDataFrame([(1, "silver", datetime(2026, 1, 2)), (1, "platinum", datetime(2026, 1, 3)), (2, "bronze", datetime(2026, 1, 2))], "customer_id int, tier string, updated_at timestamp")
feed.show()

**Expected output**

| customer_id | tier | updated_at |
| --- | --- | --- |
| 1 | platinum | 2026-01-03 00:00:00 |
| 2 | bronze | 2026-01-02 00:00:00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
latest = feed.withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("updated_at")))).where("rn = 1").drop("rn")
(DeltaTable.forPath(spark, dup_path).alias("t").merge(latest.alias("s"), "t.customer_id = s.customer_id")
 .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
result = spark.read.format("delta").load(dup_path).orderBy("customer_id")
result.show()

## Exercise 12: Read an earlier version with time travel

**Difficulty:** Easy · about 5 min · **Practises:** [Writing and reading Delta tables, history and time travel](pyspark-reference-10-reading-and-writing.ipynb#writing-and-reading-delta-tables-history-and-time-travel)

*Time travel answers "what did this table look like before the bad load?".*

Write `orders` to Delta, then overwrite it with paid orders only. Return `version` and `rows` for versions 0 and 1.

In [ ]:
tt_path = f"{WORK_DIR}/p10/delta_tt"

**Expected output**

| version | rows |
| --- | --- |
| 0 | 12 |
| 1 | 4 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
orders.write.format("delta").mode("overwrite").save(tt_path)
orders.where("status = 'paid'").write.format("delta").mode("overwrite").save(tt_path)
result = spark.createDataFrame([(v, spark.read.format("delta").option("versionAsOf", v).load(tt_path).count()) for v in [0, 1]], "version int, rows int")
result.show()

## Exercise 13: Replace one day in a Delta table

**Difficulty:** Medium · about 10 min · **Practises:** [Replace part of a table: `replaceWhere`](pyspark-reference-10-reading-and-writing.ipynb#replace-part-of-a-table-replacewhere)

*Delta's idempotent partition reload.*

Write `orders` with a `day` column to Delta partitioned by `day`. Replace day 2026-01-05 with a single corrected row (order 1 with amount 125.00) using `replaceWhere`. Return `day` and `orders`, ordered by `day`.

In [ ]:
rw_path = f"{WORK_DIR}/p10/delta_rw"

**Expected output**

| day | orders |
| --- | --- |
| 2026-01-05 | 1 |
| 2026-01-06 | 1 |
| 2026-01-07 | 2 |
| 2026-01-09 | 1 |
| 2026-01-10 | 1 |
| 2026-01-12 | 2 |
| 2026-01-13 | 1 |
| 2026-01-14 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
daily = orders.withColumn("day", F.to_date("order_ts"))
daily.write.format("delta").mode("overwrite").partitionBy("day").save(rw_path)
fixed = daily.where("order_id = 1").withColumn("amount", F.lit(D("125.00")).cast("decimal(12,2)"))  # match the table type
fixed.write.format("delta").mode("overwrite").option("replaceWhere", "day = '2026-01-05'").save(rw_path)
result = spark.read.format("delta").load(rw_path).groupBy("day").agg(F.count("*").alias("orders")).orderBy("day")
result.show()

## Exercise 14: Evolve a Delta schema on purpose

**Difficulty:** Medium · about 10 min · **Practises:** [Schema enforcement and evolution](pyspark-reference-10-reading-and-writing.ipynb#schema-enforcement-and-evolution)

*Schema enforcement blocks accidental changes; evolution allows deliberate ones.*

Write `customers` (id and name) to Delta, then try to append a row with an extra `tier` column, first without and then with `mergeSchema`. Return `attempt` and `succeeded` for both tries, in that order.

In [ ]:
evo_path = f"{WORK_DIR}/p10/delta_evo"
customers.select("customer_id", "name").write.format("delta").mode("overwrite").save(evo_path)
extra = spark.createDataFrame([(9, "India Imports", "gold")], "customer_id int, name string, tier string")

**Expected output**

| attempt | succeeded |
| --- | --- |
| plain_append | false |
| merge_schema | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def try_append(**options):
    spark.sparkContext.setLogLevel("OFF")
    try:
        extra.write.format("delta").mode("append").options(**options).save(evo_path)
        return True
    except Exception:
        return False
    finally:
        spark.sparkContext.setLogLevel("ERROR")

result = spark.createDataFrame([("plain_append", try_append()), ("merge_schema", try_append(mergeSchema="true"))], "attempt string, succeeded boolean")
result.show()

**Next:** look anything up in [PySpark reference 10: reading and writing data](pyspark-reference-10-reading-and-writing.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).